# Architecture du firmware C & portage Python → C — CL-Embedded

**Projet** : Apprentissage Incrémental pour Systèmes Embarqués à Ressources Limitées
**Carte** : NUCLEO-F439ZI (Cortex-M4 @ 180 MHz, 256 Ko SRAM, FPU FP32, *pas de NPU*)
**Auteur** : Léonard Rivals — ISAE-SUPAERO / ENAC / Edge Spectrum

---

Ce notebook est un **support visuel pédagogique**. Il explique :

1. **Quel fichier C appelle lequel** pendant l'exécution de la pipeline embarquée.
2. **Le flux d'exécution** d'une trame UART (protocole v3, dispatch par *nibble*).
3. **Comment les modèles Python existants ont été portés en C** (`export_weights_c.py` → header `.h`).
4. **Pour chaque modèle** : *où il est entraîné* (PC ou carte), comment se passe l'**inférence
   seule**, comment se passe l'**inférence + mise à jour en ligne**, et son **mécanisme
   d'adaptation** aux nouvelles tâches (forward/backprop pour les réseaux ; statistique pour
   Mahalanobis ; accumulation pour HDC).

> **Traçabilité.** Tous les extraits de code affichés sont **lus en direct** depuis les vrais
> fichiers du dépôt (via le helper `show_code`). Aucun code n'est réécrit à la main, et tous les
> chiffres de RAM / latence proviennent de `CLAUDE.md` et `docs/triple_gap.md` (campagnes carte
> réelles).

## S0 — Setup

On se replace à la racine du dépôt, on prépare le dossier de figures, et on définit le helper
`show_code(path, start, end)` qui affiche un extrait **réel** d'un fichier source avec coloration.

In [1]:
from pathlib import Path
import os, sys, textwrap

import matplotlib
matplotlib.use("Agg")  # backend non interactif (exécution nbconvert)
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle
import numpy as np
import pandas as pd

try:
    import networkx as nx
except ImportError:
    raise SystemExit("networkx requis :  pip install networkx  (ou  pip install -e '.[dev]')")

from IPython.display import Markdown, display, Code

# --- Repositionnement racine dépôt (robuste : depuis notebooks/ ou racine) ---
_cwd = Path(".").resolve()
if _cwd.name == "notebooks":
    os.chdir(_cwd.parent)
REPO_ROOT = Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

FW   = REPO_ROOT / "firmware" / "stm32f4_blink"
SRC  = FW / "src"
INC  = FW / "inc"
FIGURE_DIR = REPO_ROOT / "notebooks" / "figures" / "firmware_arch"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 120, "font.size": 11, "savefig.bbox": "tight"})

def save(fig, name):
    out = FIGURE_DIR / name
    fig.savefig(out, dpi=150)
    print(f"✓ figure : {out.relative_to(REPO_ROOT)}")

def show_code(relpath, start=1, end=None, header=True):
    '''Affiche un extrait REEL d'un fichier du dépôt (lignes [start, end], 1-indexé).'''
    p = REPO_ROOT / relpath
    lines = p.read_text(encoding="utf-8", errors="replace").splitlines()
    end = end or len(lines)
    snippet = "\n".join(lines[start - 1:end])
    if header:
        display(Markdown(f"**`{relpath}`** — lignes {start}–{end}"))
    lang = "c" if p.suffix in (".c", ".h") else "python"
    display(Code(snippet, language=lang))

print("REPO_ROOT :", REPO_ROOT)
print("Firmware  :", FW.relative_to(REPO_ROOT))
print("Figures   :", FIGURE_DIR.relative_to(REPO_ROOT))

REPO_ROOT : /home/leonard/Documents/ENAC/cl-embedded
Firmware  : firmware/stm32f4_blink
Figures   : notebooks/figures/firmware_arch


## S1 — Vue d'ensemble matérielle & contraintes

Le développement firmware cible la **NUCLEO-F439ZI** (la STM32N6 d'origine n'était pas
disponible). C'est un **Cortex-M4 avec FPU matérielle FP32**, mais **sans NPU** : le *forward
pass* **et** la *backpropagation* s'exécutent tous deux sur le CPU en FP32.

Le projet vise à combler simultanément le **triple gap** :

| Gap | Description | Statut |
|-----|-------------|--------|
| **Gap 1** | Données industrielles réelles de séries temporelles | ✅ CWRU, Pronostia, CMAPSS, Paderborn |
| **Gap 2** | Opération sous contrainte RAM avec **latence ≤ 100 ms** (chiffres mesurés) | ✅ DWT : 5–650 µs ≪ 100 ms |
| **Gap 3** | Quantification INT8 *pendant* l'apprentissage incrémental | ✅ EWC INT8 + HDC INT8 |

Le PC (Python) **entraîne** et **exporte** les modèles ; la carte **infère** et, pour certains
modèles, **se met à jour en ligne**. La communication se fait par **UART** (trame binaire v3).

In [2]:
# --- Diagramme bloc : PC <-> carte via UART ---
fig, ax = plt.subplots(figsize=(11, 4.2))
ax.axis("off")

def box(x, y, w, h, label, fc, ax=ax):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.08",
                                fc=fc, ec="#333", lw=1.5))
    ax.text(x + w/2, y + h/2, label, ha="center", va="center", fontsize=10)

box(0.2, 1.2, 3.0, 2.2,
    "PC (Python / PyTorch)\n\n• entraîne les modèles\n• calcule Fisher, μ, Σ⁻¹\n"
    "• export_weights_c.py\n• sensor_stream.py (joue les données)", "#cfe8ff")
box(7.0, 1.2, 3.6, 2.2,
    "NUCLEO-F439ZI (C, Cortex-M4)\n\n• inférence (forward)\n• mise à jour en ligne (SGD/EMA)\n"
    "• métriques online (DWT, .bss)\n• 256 Ko SRAM, FP32, pas de NPU", "#d7f4d7")

# Flèches UART
ax.add_patch(FancyArrowPatch((3.25, 2.7), (6.95, 2.7), arrowstyle="-|>",
                             mutation_scale=18, lw=2, color="#1565c0"))
ax.text(5.1, 2.95, "trame capteur (features + label + FLAGS)", ha="center", fontsize=9, color="#1565c0")
ax.add_patch(FancyArrowPatch((6.95, 1.9), (3.25, 1.9), arrowstyle="-|>",
                             mutation_scale=18, lw=2, color="#2e7d32"))
ax.text(5.1, 1.55, "réponse (pred, conf, latence, métriques)", ha="center", fontsize=9, color="#2e7d32")
ax.text(5.1, 2.3, "UART @ ST-LINK", ha="center", fontsize=9, style="italic", color="#555")

ax.set_xlim(0, 10.8); ax.set_ylim(0.8, 3.8)
ax.set_title("Architecture de travail : PC entraîne/exporte · carte infère/s'adapte", fontsize=12)
save(fig, "s1_pc_board_overview.png"); plt.show()

✓ figure : notebooks/figures/firmware_arch/s1_pc_board_overview.png


/tmp/ipykernel_36541/2414421447.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  save(fig, "s1_pc_board_overview.png"); plt.show()


## S2 — Graphe de dépendances des fichiers C

Quel fichier `#include` (et appelle) lequel ? `main.c` est le point d'entrée ; il lance
`pipeline.c`, **le dispatcher central** qui inclut tous les modèles et toute l'infrastructure
(profiling, métriques, ring buffer). Chaque modèle a son `.c`/`.h` et, pour la plupart, une
variante quantifiée (`*_int8.c`, `mahalanobis_q15.c`).

In [3]:
# Dépendances #include réelles (extraites des en-têtes de chaque .c).
# Orientation : A -> B  signifie  "A inclut / appelle B".
edges = [
    ("main.c", "pipeline.c"), ("main.c", "hw_info.c"), ("main.c", "profiling.c"),
    # pipeline = dispatcher central
    ("pipeline.c", "ewc_head.c"), ("pipeline.c", "ewc_head_int8.c"),
    ("pipeline.c", "ewc_head_regression.c"), ("pipeline.c", "ewc_head_multiclass.c"),
    ("pipeline.c", "hdc.c"), ("pipeline.c", "hdc_int8.c"),
    ("pipeline.c", "mahalanobis.c"), ("pipeline.c", "mahalanobis_q15.c"),
    ("pipeline.c", "tinyol.c"), ("pipeline.c", "tinyol_int8.c"),
    ("pipeline.c", "meta_head.c"),
    ("pipeline.c", "profiling.c"), ("pipeline.c", "metrics.c"), ("pipeline.c", "ring_buffer.c"),
    # dépendances internes entre modèles
    ("ewc_head_int8.c", "ewc_head.c"), ("tinyol_int8.c", "tinyol.c"),
    ("mahalanobis_q15.c", "mahalanobis.c"),
    ("tinyol.c", "model_weights.h"), ("meta_head.c", "meta_weights.h"),
]

role = {
    "main.c": "entrée", "pipeline.c": "dispatcher",
    "profiling.c": "infra", "metrics.c": "infra", "ring_buffer.c": "infra", "hw_info.c": "infra",
    "model_weights.h": "header généré", "meta_weights.h": "header généré",
}
for n in ["ewc_head.c","ewc_head_int8.c","ewc_head_regression.c","ewc_head_multiclass.c",
          "hdc.c","hdc_int8.c","mahalanobis.c","mahalanobis_q15.c","tinyol.c","tinyol_int8.c",
          "meta_head.c"]:
    role[n] = "modèle"
COLOR = {"entrée":"#ffd54f","dispatcher":"#ff8a65","modèle":"#90caf9",
         "infra":"#a5d6a7","header généré":"#ce93d8"}

G = nx.DiGraph(); G.add_edges_from(edges)

# Layout par couches (rangs verticaux selon le rôle)
rank = {"entrée":0,"dispatcher":1,"modèle":2,"infra":3,"header généré":3}
layers = {}
for n in G.nodes():
    layers.setdefault(rank.get(role.get(n,"modèle"),2), []).append(n)
pos = {}
for r, nodes in layers.items():
    nodes = sorted(nodes)
    for i, n in enumerate(nodes):
        pos[n] = (i - (len(nodes)-1)/2.0, -r)

fig, ax = plt.subplots(figsize=(15, 8)); ax.axis("off")
node_colors = [COLOR[role.get(n,"modèle")] for n in G.nodes()]
nx.draw_networkx_edges(G, pos, ax=ax, edge_color="#777", arrows=True,
                       arrowsize=13, width=1.2, connectionstyle="arc3,rad=0.04")
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors, node_size=2600,
                       edgecolors="#333", linewidths=1.2)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(fc=c, ec="#333", label=k) for k,c in COLOR.items()],
          loc="lower center", ncol=5, fontsize=9, frameon=True)
ax.set_title("Graphe de dépendances du firmware  (A → B : A inclut/appelle B)", fontsize=13)
save(fig, "s2_c_dependency_graph.png"); plt.show()

✓ figure : notebooks/figures/firmware_arch/s2_c_dependency_graph.png


/tmp/ipykernel_36541/165081973.py:55: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  save(fig, "s2_c_dependency_graph.png"); plt.show()


**Lecture.** `main.c` (jaune) appelle `pipeline.c` (orange), l'unique dispatcher. Celui-ci
tire **tous** les modèles (bleu) et l'infrastructure (vert). Les variantes quantifiées
réutilisent leur base FP32 (`ewc_head_int8.c → ewc_head.c`, etc.). Les **headers générés**
(violet, ex. `model_weights.h`) sont produits par les scripts d'export Python — voir **S4**.

In [4]:
show_code("firmware/stm32f4_blink/src/main.c", 1, 45)

**`firmware/stm32f4_blink/src/main.c`** — lignes 1–45

/**
 * STM32F439ZI — Pipeline anomaly detection (S1603)
 *
 * Séquence au démarrage :
 *   1. hw_clock_init()   : HSI → PLL → SYSCLK 180 MHz
 *   2. hw_uart_init()    : USART3 @ 115200, PD8=TX PD9=RX (ST-LINK VCP)
 *   3. hw_info_collect / hw_info_print + hw_dwt_calibrate : rapport UART
 *   4. pipeline_init()   : GPIO PA5, détecteur Mahalanobis (poids Flash)
 *   5. Boucle : pipeline_run() — UART frame → inférence → réponse 9 B
 *
 * Sortie UART lisible sur : minicom -b 115200 -D /dev/ttyACM0
 */

#include "stm32f4xx.h"
#include "hw_info.h"
#include "pipeline.h"
#include "profiling.h"

int main(void)
{
    /* ── 1. Horloge système → 180 MHz ───────────────────────────────── */
    hw_clock_init();

    /* ── 2. UART3 @ 115200 (PD8/PD9 = ST-LINK VCP) ─────────────────── */
    hw_uart_init();

    /* ── 3. Rapport hardware + calibration DWT ──────────────────────── */
    /* MEM: HWInfo = 36 B @ FP32 — libéré en sortie de bloc */
    {
        HWInfo info;
        hw_info_collect(&info);
        hw_info_print(&info);
        hw_dwt_calibrate(info.sysclk_hz);
    }

    /* ── 4. Init pipeline + profiling ──────────────────────────────── */
    profiling_init();
    pipeline_init();

    /* ── 5. Boucle d'inférence (bloque sur trame UART) ──────────────── */
    while (1) {
        pipeline_run();
    }
}

## S3 — Flux d'exécution de la pipeline

### 3.1 La trame UART v3

Toutes les données capteur arrivent dans une **trame binaire little-endian** :

In [5]:
# --- Schéma "ruban d'octets" de la trame v3 ---
fields = [("MAGIC", 2, "#ffe082"), ("VER", 1, "#ffe082"), ("TASK", 1, "#b3e5fc"),
          ("TIMESTAMP_MS", 4, "#b3e5fc"), ("N", 1, "#c8e6c9"),
          ("features  f32 × N", 16, "#a5d6a7"), ("label", 1, "#f8bbd0"),
          ("FLAGS", 1, "#ff8a65"), ("CRC8", 1, "#e0e0e0")]
fig, ax = plt.subplots(figsize=(14, 2.4)); ax.axis("off")
x = 0.0
for name, w, c in fields:
    ax.add_patch(Rectangle((x, 0), w, 1, fc=c, ec="#333", lw=1.3))
    ax.text(x + w/2, 0.5, name, ha="center", va="center", fontsize=9)
    ax.text(x + w/2, -0.32, f"{w} o", ha="center", va="center", fontsize=8, color="#555")
    x += w
ax.text(x/2, 1.45, "Trame capteur — protocole UART v3 (little-endian)", ha="center", fontsize=12)
ax.annotate("octet de mode\n(dispatch par nibble)", xy=(x-2.5, 1.0), xytext=(x-2.5, 2.0),
            ha="center", fontsize=8.5, color="#bf360c",
            arrowprops=dict(arrowstyle="-|>", color="#bf360c"))
ax.set_xlim(-0.5, x + 0.5); ax.set_ylim(-0.6, 2.4)
save(fig, "s3_uart_frame_v3.png"); plt.show()

✓ figure : notebooks/figures/firmware_arch/s3_uart_frame_v3.png


/tmp/ipykernel_36541/2926331928.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  save(fig, "s3_uart_frame_v3.png"); plt.show()


### 3.2 Les modes — dispatch par *nibble*

L'octet **FLAGS** encode le mode. Le **nibble haut** (4 bits de poids fort) sélectionne le
modèle/scénario, le nibble bas porte des bits d'action (`UPDATE`, `CONSOLIDATE`, `RESET`).
`pipeline_run()` teste les nibbles **dans un ordre précis** pour éviter les collisions de bits
(ex. `0xF0 & 0x30 == 0x30` : Maha-Q15 doit être testé **avant** multiclass).

In [6]:
modes = pd.DataFrame([
    ["0x10", "EWC binaire",          "ewc_forward / ewc_sgd_step",        "V3 (23 o)"],
    ["0x20", "HDC",                   "hdc_encode / hdc_update",           "V3 (23 o)"],
    ["0x30", "EWC multi-classe",      "ewc_mc_forward",                    "V3 (23 o)"],
    ["0x40", "EWC INT8",             "ewc_int8_forward / _update (Q7)",   "V3 (23 o)"],
    ["0x50", "EWC régression (RUL)",  "ewc_reg_predict (MSE)",             "V3 (23 o)"],
    ["0x60", "HDC INT8",             "hdc_int8_encode / _update",         "V3 (23 o)"],
    ["0x70", "DUAL  (RUL + faute)",   "ewc_reg + ewc_mc",                  "DUAL (25 o)"],
    ["0x80", "TinyOL (autoencodeur)", "tinyol_encode/decode + MSE",        "V3 (23 o)"],
    ["0x90", "PAIR  Maha + EWC",      "maha_score + ewc_forward",          "PAIR (22 o)"],
    ["0xA0", "PAIR  Maha + HDC",      "maha_score + hdc_predict",          "PAIR (22 o)"],
    ["0xB0", "PAIR  Maha + TinyOL",   "maha_score + tinyol",               "PAIR (22 o)"],
    ["0xC0", "TinyOL INT8 + tête OtO","tinyol_int8 + oto_int8_update",     "V3 (23 o)"],
    ["0xD0", "TRIPLE  Maha+EWC+Méta", "+ meta_forward (stacking)",         "TRIPLE (27 o)"],
    ["0xE0", "TRIPLE  Maha+HDC+Méta", "+ meta_forward (stacking)",         "TRIPLE (27 o)"],
    ["0xF0", "Mahalanobis Q15",       "maha_q15_score (Σ⁻¹ int16)",        "V3 (23 o)"],
], columns=["nibble", "mode", "fonctions C clés", "réponse"])

display(Markdown("**Bits d'action (nibble bas)** : `0x01 UPDATE` · `0x04 CONSOLIDATE` · "
                 "`0x08 RESET` — combinables avec le mode."))
display(modes)

**Bits d'action (nibble bas)** : `0x01 UPDATE` · `0x04 CONSOLIDATE` · `0x08 RESET` — combinables avec le mode.

,nibble,mode,fonctions C clés,réponse
0,0x10,EWC binaire,ewc_forward / ewc_sgd_step,V3 (23 o)
1,0x20,HDC,hdc_encode / hdc_update,V3 (23 o)
2,0x30,EWC multi-classe,ewc_mc_forward,V3 (23 o)
3,0x40,EWC INT8,ewc_int8_forward / _update (Q7),V3 (23 o)
4,0x50,EWC régression (RUL),ewc_reg_predict (MSE),V3 (23 o)
5,0x60,HDC INT8,hdc_int8_encode / _update,V3 (23 o)
6,0x70,DUAL (RUL + faute),ewc_reg + ewc_mc,DUAL (25 o)
7,0x80,TinyOL (autoencodeur),tinyol_encode/decode + MSE,V3 (23 o)
8,0x90,PAIR Maha + EWC,maha_score + ewc_forward,PAIR (22 o)
9,0xA0,PAIR Maha + HDC,maha_score + hdc_predict,PAIR (22 o)


### 3.3 La cascade de dispatch dans `pipeline_run()`

Voici le flux logique, du décodage de trame à la réponse :

In [7]:
# --- Flowchart pipeline_run() ---
fig, ax = plt.subplots(figsize=(13, 8.5)); ax.axis("off")
def fbox(x, y, w, h, label, fc):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.06",
                                fc=fc, ec="#333", lw=1.4))
    ax.text(x + w/2, y + h/2, label, ha="center", va="center", fontsize=9)
def arr(x1, y1, x2, y2, txt=None, col="#444"):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle="-|>", mutation_scale=15,
                                 lw=1.5, color=col))
    if txt: ax.text((x1+x2)/2 + 0.25, (y1+y2)/2, txt, fontsize=8, color=col)

fbox(3.7, 9.2, 4.0, 0.7, "uart_receive_sample()  — sync MAGIC + CRC8", "#ffe082")
fbox(3.7, 8.2, 4.0, 0.7, "ring_buffer_push()  (buffer de streaming)", "#c8e6c9")
fbox(3.7, 7.2, 4.0, 0.7, "lire octet FLAGS → nibble de mode", "#ff8a65")
fbox(0.3, 5.6, 3.0, 0.8, "RESET ?  → réinit modèle\n→ réponse V3", "#e0e0e0")
fbox(3.7, 5.6, 4.0, 0.8, "DISPATCH par nibble\n(0xF0→0xD0→0x90→0x70→0x30…)", "#ff8a65")
fbox(8.4, 5.6, 3.0, 0.8, "défaut : Mahalanobis\n(fallback)", "#90caf9")
# modèles
fbox(0.2, 3.9, 2.3, 0.8, "forward / score\n(inférence)", "#90caf9")
fbox(2.9, 3.9, 2.5, 0.8, "si UPDATE :\nSGD / EMA / accumulate", "#a5d6a7")
fbox(5.8, 3.9, 2.6, 0.8, "si CONSOLIDATE :\nFisher EMA / binarize", "#a5d6a7")
fbox(8.7, 3.9, 2.5, 0.8, "profiling_start/stop\n(DWT latence)", "#fff59d")
fbox(3.0, 2.3, 5.0, 0.8, "métriques online (metrics.c) :\nacc · AUROC · forgetting · RMSE · F1", "#c8e6c9")
fbox(3.4, 1.0, 4.2, 0.8, "encoder réponse → UART\n(V3 / DUAL / PAIR / TRIPLE)", "#2e7d32")

arr(5.7, 9.2, 5.7, 8.9); arr(5.7, 8.2, 5.7, 7.9); arr(5.7, 7.2, 5.7, 6.4)
arr(3.7, 6.0, 3.3, 6.0, "RESET"); arr(7.7, 6.0, 8.4, 6.0, "aucun mode")
arr(5.7, 5.6, 3.5, 4.7); arr(5.7, 5.6, 7.1, 4.7)
arr(1.8, 5.6, 1.3, 4.7)
arr(5.5, 3.9, 5.5, 3.1); arr(5.5, 2.3, 5.5, 1.8)
ax.set_xlim(0, 11.6); ax.set_ylim(0.7, 10.1)
ax.set_title("pipeline_run() — du décodage de trame à la réponse UART", fontsize=13)
save(fig, "s3_pipeline_flow.png"); plt.show()

✓ figure : notebooks/figures/firmware_arch/s3_pipeline_flow.png


/tmp/ipykernel_36541/485091131.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  save(fig, "s3_pipeline_flow.png"); plt.show()


Extrait réel de la **cascade de dispatch** (l'ordre des tests évite les collisions de bits) :

In [8]:
# La cascade de dispatch dans pipeline_run() (on localise dynamiquement les bornes).
_p = (SRC / "pipeline.c").read_text().splitlines()
_start = next(i for i,l in enumerate(_p, 1) if "PROTO_FLAG" in l and ("MAHA_Q15" in l or "0xF0" in l))
show_code("firmware/stm32f4_blink/src/pipeline.c", max(1,_start-2), _start+22)

**`firmware/stm32f4_blink/src/pipeline.c`** — lignes 640–664

* (masque 0xF0). Réponse V3 (23 B) — aucun nouveau format. mu reste fixe (pas d'EMA :
     * mu quantifié), parité avec MahalanobisDetectorInt8.anomaly_score_q15 (PC). */
    if ((uint8_t)(g_recv_flags & PROTO_PAIR_MODE_MASK) == PROTO_FLAG_MAHA_Q15) {
        normalize_zscore(raw, MAHA_DIM);
        float score   = maha_q15_score(&g_maha_q15, raw);
        int   anomaly = (score > g_maha_q15.threshold) ? 1 : 0;
        led_set(anomaly ? LED_ON : LED_OFF);
        confidence = 1.0f / (1.0f + score);

        if (g_recv_flags & PROTO_FLAG_CONSOLIDATE)
            g_current_task_id = g_recv_task_id;
        auroc_update(&g_auroc, score, (int)g_recv_label);
        acc_update(&g_acc, anomaly, (int)g_recv_label);
        fgt_update(&g_fgt, g_current_task_id, acc_compute(&g_acc));

        profiling_stop();

        MetricsSnapshot snap;
        snap.accuracy   = acc_compute(&g_acc);
        snap.auroc      = auroc_compute(&g_auroc);
        snap.forgetting = fgt_avg_forgetting(&g_fgt);
        uart_send_response_v3((uint8_t)anomaly, confidence,
                              profiling_get_latency_us(), PROTO_STATUS_OK, &snap);
        energy_marker_phase(PHASE_IDLE);   /* S3304 */
        return;

### 3.4 Les quatre formats de réponse

Selon le mode, la carte renvoie un des 4 layouts (tous incluent la **latence DWT** mesurée) :

In [9]:
layouts = {
 "V3 (23 o) — modèle simple": [("pred",1),("conf",4),("lat_us",4),("ram_b",2),("acc",4),("auroc",4),("forget",4)],
 "DUAL (25 o) — RUL + faute":  [("pred_faute",1),("conf",4),("rul",4),("lat_us",4),("f1",4),("rmse",4),("forget",4)],
 "PAIR (22 o) — Maha+sup.":    [("pred_maha",1),("score_maha",4),("pred_sup",1),("conf_sup",4),("lat_us",4),("auroc",4),("f1",4)],
 "TRIPLE (27 o) — +méta":      [("pred_maha",1),("score_maha",4),("pred_sup",1),("conf_sup",4),("lat_us",4),("auroc",4),("f1",4),("pred_méta",1),("prob_méta",4)],
}
fig, axes = plt.subplots(4, 1, figsize=(14, 7));
palette = ["#ffcc80","#ffe082","#c5e1a5","#80deea","#b39ddb","#f48fb1","#bcaaa4","#90caf9","#a5d6a7"]
for ax, (title, fl) in zip(axes, layouts.items()):
    ax.axis("off"); x=0
    for i,(name,w) in enumerate(fl):
        ax.add_patch(Rectangle((x,0), w, 1, fc=palette[i%len(palette)], ec="#333", lw=1.1))
        ax.text(x+w/2, 0.5, f"{name}\n{w}o", ha="center", va="center", fontsize=7.5)
        x += w
    ax.set_xlim(-0.3, 28); ax.set_ylim(-0.2, 1.2)
    ax.set_title(title, fontsize=10, loc="left")
fig.suptitle("Les 4 formats de réponse UART", fontsize=13)
fig.tight_layout(rect=(0,0,1,0.96))
save(fig, "s3_response_layouts.png"); plt.show()

✓ figure : notebooks/figures/firmware_arch/s3_response_layouts.png


/tmp/ipykernel_36541/762658417.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  save(fig, "s3_response_layouts.png"); plt.show()


## S4 — Le portage Python → C

C'est le cœur de la question : **comment un modèle entraîné en Python devient-il du C
embarqué ?** Le principe est *« entraîner sur PC, figer les poids dans un header C »*.

```
checkpoint Python            export_weights_c.py            header C                firmware
(.pt / .pkl / .json)   ──▶   (lecture poids + quantif)  ──▶  (tableaux static const) ──▶  #include + flash
                                                                     │
                                              test_vectors.h  ◀───────┘  (validation parité C ↔ Python)
```

Règle stricte du projet : **`model_weights.h` n'est jamais édité à la main** — toujours
régénéré par `scripts/export_weights_c.py` ou `scripts/export_weights_tinyol.py`.

In [10]:
# --- Diagramme du pipeline d'export ---
fig, ax = plt.subplots(figsize=(13, 4.6)); ax.axis("off")
def pbox(x,y,w,h,label,fc):
    ax.add_patch(FancyBboxPatch((x,y),w,h, boxstyle="round,pad=0.02,rounding_size=0.06",
                                fc=fc, ec="#333", lw=1.4))
    ax.text(x+w/2, y+h/2, label, ha="center", va="center", fontsize=9)
pbox(0.2, 2.4, 2.6, 1.2, "Entraînement PC\n(PyTorch / numpy)\ntrain_ewc.py …", "#cfe8ff")
pbox(0.4, 0.6, 2.2, 1.0, "checkpoint\n.pt / .pkl / .json", "#e1bee7")
pbox(3.4, 2.0, 3.0, 1.8, "export_weights_c.py\n\n• lit les poids\n• quantif INT8 / Q15\n"
     "• écrit tableaux C", "#fff59d")
pbox(7.0, 2.4, 2.8, 1.2, "header généré .h\nstatic const float W[][]…\n#define …_PROVIDED 1", "#ce93d8")
pbox(7.2, 0.6, 2.4, 1.0, "test_vectors.h\n(entrée + sortie attendue)", "#ffe082")
pbox(10.2, 2.4, 2.4, 1.2, "firmware C\n#include + flash\nparité validée", "#d7f4d7")
arr_ = lambda a,b: ax.add_patch(FancyArrowPatch(a,b,arrowstyle="-|>",mutation_scale=16,lw=1.6,color="#444"))
arr_((1.5,2.4),(1.5,1.65)); arr_((2.6,1.1),(3.4,2.4)); arr_((6.4,2.9),(7.0,2.9))
arr_((6.4,2.4),(7.2,1.3)); arr_((9.8,2.9),(10.2,2.9))
ax.add_patch(FancyArrowPatch((8.4,1.1),(11.4,2.4),arrowstyle="-|>",mutation_scale=16,lw=1.4,
             color="#2e7d32", connectionstyle="arc3,rad=-0.2"))
ax.text(9.7,1.5,"valide", fontsize=8, color="#2e7d32")
ax.set_xlim(0,12.8); ax.set_ylim(0.3,4.0)
ax.set_title("Pipeline de portage : Python → header C → firmware (avec validation de parité)", fontsize=12)
save(fig, "s4_export_pipeline.png"); plt.show()

✓ figure : notebooks/figures/firmware_arch/s4_export_pipeline.png


/tmp/ipykernel_36541/1084984467.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  save(fig, "s4_export_pipeline.png"); plt.show()


In [11]:
recap = pd.DataFrame([
 ["EWC",        "train_ewc.py",          ".pt",        "model_weights_ewc.h",        "FP32 / INT8 (Q7)", "✅ exacte"],
 ["Mahalanobis","train_mahalanobis.py",  ".pkl",       "model_weights.h / *_q15.h",  "FP32 / INT8 / Q15","✅ exacte"],
 ["HDC",        "train_hdc.py",          ".npz",       "— (HW-only)",                "INT8 (base vec.)", "N/A (carte)"],
 ["TinyOL",     "export_weights_tinyol.py",".pt",      "model_weights.h (section)",  "FP32 / INT8",      "N/A (archi distincte)"],
 ["Méta",       "train_meta_learner.py", ".json",      "meta_weights.h",             "FP32",             "✅ exacte"],
], columns=["modèle","script (PC)","checkpoint","header généré","quantif","parité PC↔carte"])
display(recap)

,modèle,script (PC),checkpoint,header généré,quantif,parité PC↔carte
0,EWC,train_ewc.py,.pt,model_weights_ewc.h,FP32 / INT8 (Q7),✅ exacte
1,Mahalanobis,train_mahalanobis.py,.pkl,model_weights.h / *_q15.h,FP32 / INT8 / Q15,✅ exacte
2,HDC,train_hdc.py,.npz,— (HW-only),INT8 (base vec.),N/A (carte)
3,TinyOL,export_weights_tinyol.py,.pt,model_weights.h (section),FP32 / INT8,N/A (archi distincte)
4,Méta,train_meta_learner.py,.json,meta_weights.h,FP32,✅ exacte


Voici **comment** le script écrit un tableau C à partir d'un tenseur PyTorch — extrait réel
de `export_weights_c.py` (fonction d'export de la tête EWC compatible carte) :

In [12]:
# Localise la fonction d'export de la tête EWC dans le script.
_e = (REPO_ROOT / "scripts/export_weights_c.py").read_text().splitlines()
_s = next((i for i,l in enumerate(_e,1) if "def export_ewc_head_board_to_c" in l), 1)
show_code("scripts/export_weights_c.py", _s, _s+34)

**`scripts/export_weights_c.py`** — lignes 187–221

def export_ewc_head_board_to_c(model_path: Path, out_path: Path) -> None:
    """Génère inc/model_weights_ewc.h depuis un checkpoint EWCMlpMulticlass 5→32→16→2.

    Cible la tête EWC single-mode du firmware (``g_ewc_head``, EWC_IN=5, EWC_OUT=2).
    L'architecture EWCMlpMulticlass(input_dim=5, n_classes=2, hidden=[32,16]) est
    bit-pour-bit équivalente à ``ewc_forward`` (relu→relu→logits→argmax), d'où la
    parité board↔PC. nn.Linear.weight est [out, in] = layout board w[out][in].

    Le header émet ``#define EWC_HEAD_WEIGHTS_PROVIDED 1`` qui active le chargement
    Flash dans ``pipeline_init`` (sinon : init Xavier historique).
    """
    import torch  # noqa: PLC0415

    checkpoint = torch.load(model_path, map_location="cpu")
    sd = checkpoint.get("model_state_dict", checkpoint)

    def _get(key: str) -> np.ndarray:
        return sd[key].detach().cpu().numpy().astype(np.float32)

    w1, b1 = _get("fc1.weight"), _get("fc1.bias")   # [32, IN], [32]
    w2, b2 = _get("fc2.weight"), _get("fc2.bias")   # [16,32], [16]
    w3, b3 = _get("fc3.weight"), _get("fc3.bias")   # [2,16], [2]

    # IN est configurable au build (S3506/S3507 : `make EWC_IN=k`). Seules les couches
    # cachées sont figées (EWC_H1=32, EWC_H2=16) ; la dim d'entrée k varie par condition.
    ewc_in = int(w1.shape[1])
    assert w1.shape[0] == 32, f"w1 {w1.shape} — archi board EWC_H1=32 attendu"
    assert w3.shape == (2, 16), f"w3 {w3.shape} ≠ (2,16) — archi board EWC_OUT=2, EWC_H2=16"
    for name, arr in [("w1", w1), ("b1", b1), ("w2", w2), ("b2", b2), ("w3", w3), ("b3", b3)]:
        if not np.isfinite(arr).all():  # NaN/inf → "nanf" invalide en C ; échec explicite
            raise ValueError(f"EWC {name} contient NaN/inf — entraînement board divergé, "
                             "ré-entraîner (cf. clip_grad dans train_board_reference.py)")

    lines = [
        "/**",

Et voici un extrait du **header généré** correspondant (poids figés en `static const`) :

In [13]:
show_code("firmware/stm32f4_blink/inc/model_weights_ewc.h", 1, 30)

**`firmware/stm32f4_blink/inc/model_weights_ewc.h`** — lignes 1–30

/**
 * model_weights_ewc.h — Poids tête EWC single-mode (g_ewc_head) générés.
 * Généré par scripts/export_weights_c.py --ewc-head — ne pas modifier à la main.
 * Parité : EWCMlpMulticlass(4, 2, [32,16]) == ewc_forward (FP32).
 * Dim d'entrée k=4 → builder avec `make EWC_IN=4` (S3507).
 */

#pragma once
#include "ewc_head.h"

#define EWC_HEAD_WEIGHTS_PROVIDED 1
#define EWC_HEAD_NATIVE_DIM 4   /* dim k des poids ci-dessous (cf. EWC_IN au build) */

static const float EWC_W1_INIT[32][4] = {
    {-0.15518144f, 0.09826800f, -0.36534512f, 0.01387356f},
    {-0.13989702f, 0.18367526f, -0.39831546f, -0.15681218f},
    {-0.05013122f, -0.12384982f, -0.01435369f, -0.26557016f},
    {0.37420130f, -0.46379739f, 0.27591088f, 0.21403110f},
    {-0.35262340f, 0.36482173f, 0.12172867f, 0.48696086f},
    {0.14768343f, -0.31374419f, 0.27688324f, -0.15606721f},
    {0.01305771f, 0.52793175f, 0.12209656f, -0.22683211f},
    {0.43172359f, 0.05818064f, 0.42698458f, -0.28129798f},
    {-0.26535034f, -0.19241099f, -0.45397568f, 0.28302103f},
    {0.09904735f, 0.54470587f, 0.21933000f, 0.21619901f},
    {0.35724053f, -0.35898498f, 0.01941134f, -0.32691479f},
    {0.31330648f, -0.13727206f, 0.10887478f, -0.07129655f},
    {0.33739352f, -0.28530303f, -0.33729064f, -0.25692844f},
    {0.39124429f, 0.27504906f, 0.49197680f, -0.31391063f},
    {-0.30306762f, -0.29335505f, -0.23092502f, 0.06165010f},
    {0.06468537f, 0.45915300f, -0.44640195f, -0.29989108f},

## S5 — Les modèles, un par un

Pour chaque modèle : **où il est entraîné**, **l'inférence sur carte**, **la mise à jour en
ligne sur carte**, et son **mécanisme d'adaptation**.

### 5.1 EWC — réseau de neurones régularisé (le seul vrai « apprentissage » sur carte)

**Architecture** : MLP `k → 32 → 16 → 2` (ReLU sur les couches cachées, logits en sortie).

- **Entraînement initial** : sur **PC** (`train_ewc.py`), poids exportés vers
  `model_weights_ewc.h`.
- **Inférence carte** : `ewc_forward()` — deux couches ReLU puis logits + argmax. ~50 µs (DWT).
- **Mise à jour en ligne carte** : `ewc_sgd_step()` (flag `UPDATE`) fait une **vraie
  rétropropagation FP32** sur la carte (softmax + cross-entropy), puis `ewc_consolidate()`
  (flag `CONSOLIDATE`) à la frontière de tâche. Inférence + MAJ : ~240–340 µs (≪ 100 ms, Gap 2).

**Adaptation aux nouvelles tâches — la régularisation EWC.** Pour éviter l'*oubli
catastrophique*, EWC ajoute à la perte une pénalité quadratique qui « ancre » les poids
importants pour les tâches passées :

$$\mathcal{L} = \underbrace{\text{CE}(y, \hat y)}_{\text{tâche courante}}
   + \frac{\lambda}{2}\sum_i F_i\,(\theta_i - \theta_i^\*)^2$$

où $F_i$ est la **diagonale de la matrice de Fisher** (importance du poids $i$) et
$\theta_i^\*$ le **snapshot** des poids après la tâche précédente. `ewc_consolidate()` met à
jour $F$ (EMA) et reprend le snapshot $\theta^\*$ à chaque frontière de tâche.

In [14]:
# --- Schéma forward + backprop EWC ---
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

def draw_mlp(ax, title, sizes, labels, back=False):
    ax.axis("off"); ax.set_title(title, fontsize=12)
    xs = np.linspace(0.1, 0.9, len(sizes))
    centers = []
    for li,(x,n,lab) in enumerate(zip(xs, sizes, labels)):
        ys = np.linspace(0.2, 0.8, n)
        centers.append((x, ys))
        for y in ys:
            ax.add_patch(plt.Circle((x,y), 0.02, fc="#90caf9", ec="#1565c0", zorder=3))
        ax.text(x, 0.07, lab, ha="center", fontsize=9)
    for li in range(len(sizes)-1):
        x1, ys1 = centers[li]; x2, ys2 = centers[li+1]
        for y1 in ys1:
            for y2 in ys2:
                ax.plot([x1,x2],[y1,y2], color="#bbb", lw=0.4, zorder=1)
    if back:
        for li in range(len(sizes)-1, 0, -1):
            x = (xs[li]+xs[li-1])/2
            ax.annotate("", xy=(xs[li-1]+0.02, 0.9), xytext=(xs[li]-0.02, 0.9),
                        arrowprops=dict(arrowstyle="-|>", color="#d32f2f", lw=2))
        ax.text(0.5, 0.95, "gradients : dout → dh2 → dh1  (SGD, lr=0.01)",
                ha="center", color="#d32f2f", fontsize=10)
    else:
        ax.annotate("", xy=(0.92,0.9), xytext=(0.08,0.9),
                    arrowprops=dict(arrowstyle="-|>", color="#2e7d32", lw=2))
        ax.text(0.5, 0.95, "forward : ReLU · ReLU · logits", ha="center", color="#2e7d32", fontsize=10)

draw_mlp(axes[0], "Inférence — ewc_forward()", [5,8,5,2],
         ["x  (k feat.)","h1 = ReLU(W1·x+b1)","h2 = ReLU(W2·h1+b2)","logits (2)"])
draw_mlp(axes[1], "Mise à jour — ewc_sgd_step()", [5,8,5,2],
         ["x","h1","h2","softmax+CE"], back=True)
fig.suptitle("EWC : forward (inférence) vs backprop (mise à jour en ligne sur carte)", fontsize=13)
save(fig, "s5_ewc_forward_backprop.png"); plt.show()

✓ figure : notebooks/figures/firmware_arch/s5_ewc_forward_backprop.png


/tmp/ipykernel_36541/1082787922.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  save(fig, "s5_ewc_forward_backprop.png"); plt.show()


In [15]:
show_code("firmware/stm32f4_blink/src/ewc_head.c", 71, 100)   # ewc_forward

**`firmware/stm32f4_blink/src/ewc_head.c`** — lignes 71–100

void ewc_forward(const EWCHead *h, const float *x, float *out)
{
    /* MEM: h1 = 128 B @ FP32, h2 = 64 B @ FP32 (stack local) */
    float h1[EWC_H1];
    float h2[EWC_H2];

    for (int j = 0; j < EWC_H1; j++) {
        float acc = h->b1[j];
        for (int i = 0; i < EWC_IN; i++) {
            acc += h->w1[j][i] * x[i];
        }
        h1[j] = relu(acc);
    }

    for (int j = 0; j < EWC_H2; j++) {
        float acc = h->b2[j];
        for (int i = 0; i < EWC_H1; i++) {
            acc += h->w2[j][i] * h1[i];
        }
        h2[j] = relu(acc);
    }

    for (int j = 0; j < EWC_OUT; j++) {
        float acc = h->b3[j];
        for (int i = 0; i < EWC_H2; i++) {
            acc += h->w3[j][i] * h2[i];
        }
        out[j] = acc;  /* logits bruts */
    }
}

In [16]:
show_code("firmware/stm32f4_blink/src/ewc_head.c", 116, 145)  # ewc_sgd_step (début : forward + grad sortie)

**`firmware/stm32f4_blink/src/ewc_head.c`** — lignes 116–145

void ewc_sgd_step(EWCHead *h, const float *x, int label)
{
    /* Activations forward — MEM: (128 + 64 + 8) B @ FP32 */
    float h1[EWC_H1];      /* MEM: 128 B @ FP32 */
    float h2[EWC_H2];      /* MEM:  64 B @ FP32 */
    float logits[EWC_OUT]; /* MEM:   8 B @ FP32 */

    /* ── 1. Forward ──────────────────────────────────────────────────── */
    for (int j = 0; j < EWC_H1; j++) {
        float acc = h->b1[j];
        for (int i = 0; i < EWC_IN; i++) acc += h->w1[j][i] * x[i];
        h1[j] = relu(acc);
    }
    for (int j = 0; j < EWC_H2; j++) {
        float acc = h->b2[j];
        for (int i = 0; i < EWC_H1; i++) acc += h->w2[j][i] * h1[i];
        h2[j] = relu(acc);
    }
    for (int j = 0; j < EWC_OUT; j++) {
        float acc = h->b3[j];
        for (int i = 0; i < EWC_H2; i++) acc += h->w3[j][i] * h2[i];
        logits[j] = acc;
    }

    /* ── 2. Softmax + gradient sortie (CE loss) ───────────────────────
     * dL/dlogits[j] = softmax[j] - one_hot(label)[j]              */
    float dout[EWC_OUT];   /* MEM: 8 B @ FP32 */
    float max_logit = logits[0];
    for (int j = 1; j < EWC_OUT; j++) {
        if (logits[j] > max_logit) max_logit = logits[j];

In [17]:
show_code("firmware/stm32f4_blink/src/ewc_head.c", 212, 242)  # ewc_consolidate (Fisher EMA + snapshot)

**`firmware/stm32f4_blink/src/ewc_head.c`** — lignes 212–242

void ewc_consolidate(EWCHead *h, float alpha)
{
    float one_minus_alpha = 1.0f - alpha;

    /* Couche 1 — grad² ≈ w² (proxy Fisher diagonal online, cf. Kirkpatrick2017EWC) */
    for (int j = 0; j < EWC_H1; j++) {
        for (int i = 0; i < EWC_IN; i++) {
            float g2 = h->w1[j][i] * h->w1[j][i];
            h->fisher1[j][i] = alpha * h->fisher1[j][i] + one_minus_alpha * g2;
            h->star_w1[j][i] = h->w1[j][i];
        }
    }

    /* Couche 2 */
    for (int j = 0; j < EWC_H2; j++) {
        for (int i = 0; i < EWC_H1; i++) {
            float g2 = h->w2[j][i] * h->w2[j][i];
            h->fisher2[j][i] = alpha * h->fisher2[j][i] + one_minus_alpha * g2;
            h->star_w2[j][i] = h->w2[j][i];
        }
    }

    /* Couche 3 — pas de Fisher sur les biais (standard EWC) */
    for (int j = 0; j < EWC_OUT; j++) {
        for (int i = 0; i < EWC_H2; i++) {
            float g2 = h->w3[j][i] * h->w3[j][i];
            h->fisher3[j][i] = alpha * h->fisher3[j][i] + one_minus_alpha * g2;
            h->star_w3[j][i] = h->w3[j][i];
        }
    }
}

**Variantes EWC** (même schéma, sorties/quantif différentes) :
- **EWC-INT8** (`ewc_head_int8.c`) — poids Q7, *fake-quant* : forward/backprop en FP32 puis
  re-quantification SAT8 (réponse au Gap 3).
- **EWC-Régression** (`ewc_head_regression.c`) — sortie scalaire, perte **MSE** (RUL CMAPSS).
- **EWC-Multiclasse** (`ewc_head_multiclass.c`) — `9 → 32 → 16 → 10`, cross-entropy 10 classes.

### 5.2 HDC — calcul hyperdimensionnel (non-neuronal)

**Idée** : projeter chaque échantillon dans un espace de très grande dimension (D ≈ 1000) en
vecteurs **binaires ±1**, puis maintenir une **mémoire associative** : un *prototype* accumulé
par classe.

- **Entraînement** : sur **PC** (`train_hdc.py`). Sur carte : la projection est embarquée et
  l'apprentissage se fait *en ligne* → **HW-only, parité N/A par construction** (décision
  documentée).
- **Inférence carte** : `hdc_encode()` binarise la projection, `hdc_predict()` prend l'`argmax`
  du produit scalaire avec chaque prototype.
- **Mise à jour en ligne** : `hdc_update()` **accumule** simplement l'hypervecteur dans le
  prototype de sa classe (`am[label] += hv`) ; `hdc_binarize()` re-binarise à la consolidation.

**Adaptation aux nouvelles tâches.** Pas de gradient, pas de matrice de Fisher : l'adaptation
est une **accumulation additive**. C'est *intrinsèquement résistant à l'oubli catastrophique*
(ajouter une classe ne détruit pas les prototypes existants).

In [18]:
# --- Schéma HDC ---
fig, ax = plt.subplots(figsize=(13, 4.6)); ax.axis("off")
def hbox(x,y,w,h,t,fc): ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle="round,pad=0.02,rounding_size=0.06",fc=fc,ec="#333",lw=1.3)); ax.text(x+w/2,y+h/2,t,ha="center",va="center",fontsize=9)
hbox(0.2,1.8,2.0,1.0,"x  (features)","#cfe8ff")
hbox(2.8,1.8,2.6,1.0,"projection +\nbinarisation\nhdc_encode()","#fff59d")
hbox(6.0,1.8,2.2,1.0,"hv ∈ {−1,+1}^D\n(D≈1000)","#a5d6a7")
hbox(9.0,2.9,2.4,0.9,"prédire :\nargmax ⟨hv, amₖ⟩","#90caf9")
hbox(9.0,1.0,2.4,0.9,"mettre à jour :\nam[label] += hv","#c8e6c9")
for a,b in [((2.2,2.3),(2.8,2.3)),((5.4,2.3),(6.0,2.3)),((8.2,2.3),(9.0,3.35)),((8.2,2.3),(9.0,1.45))]:
    ax.add_patch(FancyArrowPatch(a,b,arrowstyle="-|>",mutation_scale=15,lw=1.5,color="#444"))
ax.text(10.2,0.5,"accumulation additive → pas d'oubli catastrophique",ha="center",fontsize=9,style="italic",color="#555")
ax.set_xlim(0,12); ax.set_ylim(0.2,4.1)
ax.set_title("HDC : encodage hyperdimensionnel + mémoire associative", fontsize=12)
save(fig, "s5_hdc_scheme.png"); plt.show()

✓ figure : notebooks/figures/firmware_arch/s5_hdc_scheme.png


/tmp/ipykernel_36541/1524649177.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  save(fig, "s5_hdc_scheme.png"); plt.show()


In [19]:
show_code("firmware/stm32f4_blink/src/hdc.c", 12, 42)   # hdc_encode + hdc_predict

**`firmware/stm32f4_blink/src/hdc.c`** — lignes 12–42

void hdc_encode(const HDCClassifier *h, const float *x, float *hv_out)
{
    /* hv_out[i] = sign(proj[i] · x) ∈ {-1.0f, +1.0f}
     * Propriété : sum(hv_out[i]²) == HDC_DIM car chaque élément est ±1. */
    for (int i = 0; i < HDC_DIM; i++) {
        float dot = 0.0f;
        for (int j = 0; j < HDC_N_FEATURES; j++) {
            dot += h->proj[i][j] * x[j];
        }
        hv_out[i] = (dot >= 0.0f) ? 1.0f : -1.0f;
    }
}

int hdc_predict(const HDCClassifier *h, const float *hv)
{
    /* argmax_c dot(am[c], hv) — proxy cosinus pour hv binarisé */
    int   best_class = 0;
    float best_score = 0.0f;

    for (int c = 0; c < HDC_N_CLASSES; c++) {
        float score = 0.0f;
        for (int i = 0; i < HDC_DIM; i++) {
            score += h->am[c][i] * hv[i];
        }
        if (c == 0 || score > best_score) {
            best_score = score;
            best_class = c;
        }
    }
    return best_class;
}

In [20]:
show_code("firmware/stm32f4_blink/src/hdc.c", 44, 60)   # hdc_update (accumulation)

**`firmware/stm32f4_blink/src/hdc.c`** — lignes 44–60

void hdc_update(HDCClassifier *h, const float *hv, int label)
{
    /* Accumulation pure — pas de recalcul depuis zéro, pas d'oubli catastrophique. */
    for (int i = 0; i < HDC_DIM; i++) {
        h->am[label][i] += hv[i];
    }
    h->n_trained++;
}

/**
 * hdc_update_with_sample — Update AM + stocke l'échantillon brut dans le buffer.
 *
 * Stockage en uint8 pour économiser la RAM :
 *   raw = (uint8_t)((x[j] + 1.0f) * 127.5f)  — plage [-1,1] → [0,255]
 * MEM buf_storage : HDC_RETRAIN_BUF * (HDC_N_FEATURES + 1) B = 50 * 6 = 300 B
 *   (features quantifiées + label entrelacés, géré par ring_buffer — S3402)
 */

### 5.3 Mahalanobis — détecteur d'anomalie statistique

**Idée** : modéliser les données *normales* par une gaussienne $(\mu, \Sigma)$ et scorer la
distance de Mahalanobis $d(x) = \sqrt{(x-\mu)^\top \Sigma^{-1} (x-\mu)}$. Anomalie si
$d > $ seuil.

- **Entraînement** : sur **PC** (`train_mahalanobis.py`, `fit_task`). $\mu$ et $\Sigma^{-1}$
  sont calculés offline et exportés (`model_weights.h`). Parité carte↔PC **exacte**.
- **Inférence carte** : `maha_score()` — différence, produit matrice-vecteur, racine (FPU).
  ~5 µs (DWT).
- **Mise à jour en ligne** : `maha_update()` fait une **EMA sur $\mu$ uniquement** ;
  $\Sigma^{-1}$ reste **figé** (recalculer une inverse de covariance sur MCU serait trop coûteux).

**Variante Q15** (`mahalanobis_q15.c`, Sprint 34) : $\Sigma^{-1}$ stockée en **int16 Q15** plutôt
qu'INT8, car la grande dynamique de $\Sigma^{-1}$ écrasait la précision INT8 (ΔAUROC dégradé).
La distance est recalculée en FP32 après déquantification → parité bit-à-bit avec Python.

In [21]:
# --- Schéma Mahalanobis (ellipses + score) ---
fig, ax = plt.subplots(figsize=(7.5, 6))
rng = np.random.default_rng(42)
cov = np.array([[1.0,0.6],[0.6,0.8]]); mu = np.array([0,0])
pts = rng.multivariate_normal(mu, cov, 300)
ax.scatter(pts[:,0], pts[:,1], s=10, c="#90caf9", label="données normales (PC : fit μ, Σ)")
from matplotlib.patches import Ellipse
vals, vecs = np.linalg.eigh(cov); ang = np.degrees(np.arctan2(*vecs[:,0][::-1]))
for k in (1,2,3):
    ax.add_patch(Ellipse(mu, *(2*k*np.sqrt(vals)), angle=ang, fill=False, ec="#1565c0", lw=1.2, ls="--"))
anom = np.array([3.2, -2.0])
ax.scatter(*anom, s=120, c="#d32f2f", marker="*", zorder=5, label="anomalie : d > seuil")
ax.annotate("d(x)=√((x−μ)ᵀΣ⁻¹(x−μ))", xy=anom, xytext=(1.0, -3.2),
            arrowprops=dict(arrowstyle="-|>", color="#d32f2f"), fontsize=10, color="#d32f2f")
ax.scatter(*mu, c="k", marker="+", s=120)
ax.text(mu[0]+0.1, mu[1]+0.1, "μ", fontsize=12)
ax.legend(loc="upper left", fontsize=9); ax.set_aspect("equal")
ax.set_title("Mahalanobis : iso-contours Σ⁻¹ et score de distance", fontsize=12)
save(fig, "s5_mahalanobis_scheme.png"); plt.show()

✓ figure : notebooks/figures/firmware_arch/s5_mahalanobis_scheme.png


/tmp/ipykernel_36541/1988125136.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  save(fig, "s5_mahalanobis_scheme.png"); plt.show()


In [22]:
show_code("firmware/stm32f4_blink/src/mahalanobis.c", 39, 72)  # maha_score + maha_update (EMA sur μ)

**`firmware/stm32f4_blink/src/mahalanobis.c`** — lignes 39–72

float maha_score(const MahalanobisDetector *det, const float *x)
{
    float diff[MAHA_DIM];                  /* MEM: 20 B @ FP32 */
    float left[MAHA_DIM];                  /* MEM: 20 B @ FP32 — precision @ diff */

    for (int i = 0; i < MAHA_DIM; i++) {
        diff[i] = x[i] - det->mean[i];
    }

    for (int i = 0; i < MAHA_DIM; i++) {
        float acc = 0.0f;
        for (int j = 0; j < MAHA_DIM; j++) {
            acc += det->precision[i][j] * diff[j];
        }
        left[i] = acc;
    }

    float dist_sq = 0.0f;
    for (int i = 0; i < MAHA_DIM; i++) {
        dist_sq += left[i] * diff[i];
    }

    return fpu_sqrtf(dist_sq > 0.0f ? dist_sq : 0.0f);
}

/* EMA sur mean uniquement — Σ⁻¹ reste figée (calculée offline en Python) */
void maha_update(MahalanobisDetector *det, const float *x)
{
    float alpha = det->ema_alpha;
    float one_minus = 1.0f - alpha;
    for (int i = 0; i < MAHA_DIM; i++) {
        det->mean[i] = one_minus * det->mean[i] + alpha * x[i];
    }
}

### 5.4 TinyOL — autoencodeur + tête OtO

**Idée** : un **autoencodeur** apprend à reconstruire les données normales ; une grande
**erreur de reconstruction (MSE)** signale une anomalie. Une petite **tête OtO** (One-to-One)
peut apprendre en ligne.

- **Entraînement** : encodeur pré-entraîné sur **PC** (`export_weights_tinyol.py`) puis **figé**.
  L'archi carte (`5→32→16→32→5`) diffère de l'archi PC → parité N/A par construction.
- **Inférence carte** : `tinyol_encode()` → `tinyol_decode()` → `tinyol_reconstruction_error()`
  (MSE > seuil).
- **Mise à jour en ligne** : seule la **tête OtO INT8** apprend (`oto_int8_update()`, BCE) ;
  l'autoencodeur n'est **pas** rétropropagé en bare-metal (note honnête).

In [23]:
# --- Schéma autoencodeur TinyOL ---
fig, ax = plt.subplots(figsize=(12, 4.2)); ax.axis("off")
sizes = [5,32,16,32,5]; labels=["x (5)","32","emb (16)","32","x̂ (5)"]
xs = np.linspace(0.07,0.93,5)
for x,n,lab in zip(xs,sizes,labels):
    hh = min(n,16)/16*0.6
    ax.add_patch(FancyBboxPatch((x-0.04, 0.5-hh/2), 0.08, hh, boxstyle="round,pad=0.005",
                                fc="#90caf9" if lab!='emb (16)' else "#ffab91", ec="#1565c0"))
    ax.text(x, 0.12, lab, ha="center", fontsize=9)
ax.annotate("", xy=(0.5,0.9), xytext=(0.07,0.9), arrowprops=dict(arrowstyle="-|>",color="#2e7d32",lw=2))
ax.text(0.27,0.95,"encodeur (figé)",ha="center",color="#2e7d32",fontsize=9)
ax.annotate("", xy=(0.93,0.9), xytext=(0.5,0.9), arrowprops=dict(arrowstyle="-|>",color="#1565c0",lw=2))
ax.text(0.72,0.95,"décodeur",ha="center",color="#1565c0",fontsize=9)
ax.text(0.5,-0.02,"score d'anomalie = MSE(x, x̂)   ·   tête OtO apprise en ligne sur emb",
        ha="center", fontsize=10, style="italic", color="#555")
ax.set_xlim(0,1); ax.set_ylim(-0.1,1.05)
ax.set_title("TinyOL : autoencodeur (reconstruction) + tête OtO", fontsize=12)
save(fig, "s5_tinyol_scheme.png"); plt.show()

✓ figure : notebooks/figures/firmware_arch/s5_tinyol_scheme.png


/tmp/ipykernel_36541/3260388132.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  save(fig, "s5_tinyol_scheme.png"); plt.show()


In [24]:
show_code("firmware/stm32f4_blink/src/tinyol.c", 29, 50)   # tinyol_encode

**`firmware/stm32f4_blink/src/tinyol.c`** — lignes 29–50

void tinyol_encode(const TinyOLEncoder *enc, const float *x, float *emb)
{
    float h1[TINYOL_H1];  /* MEM: 128 B @ FP32 stack local */

    /* Couche 1 : Linear(n_in→H1) + ReLU */
    for (uint32_t j = 0; j < TINYOL_H1; j++) {
        float acc = enc->b_enc1[j];
        for (uint32_t i = 0; i < TINYOL_IN; i++) {
            acc += enc->w_enc1[j][i] * x[i];
        }
        h1[j] = relu_f(acc);
    }

    /* Couche 2 : Linear(H1→EMB) + ReLU */
    for (uint32_t j = 0; j < TINYOL_EMB; j++) {
        float acc = enc->b_enc2[j];
        for (uint32_t i = 0; i < TINYOL_H1; i++) {
            acc += enc->w_enc2[j][i] * h1[i];
        }
        emb[j] = relu_f(acc);
    }
}

### 5.5 Méta-modèle — *stacking* (modes TRIPLE)

Un petit **arbitre** (régression logistique ou MLP 1 couche) combine les sorties d'une paire
**Mahalanobis + supervisé** à partir de 4 features
$[p_{\text{maha}},\ p_{\text{sup}},\ \text{désaccord},\ \text{conf}_{\text{sup}}]$.

- **Entraînement** : sur **PC** (`train_meta_learner.py`), poids figés dans `meta_weights.h`
  (parité exacte). Pas de mise à jour en ligne.
- **Inférence carte** : `meta_forward()` — sigmoïde (logreg) ou MLP, dans les modes TRIPLE
  `0xD0`/`0xE0`. Coût négligeable (4 features).

In [25]:
show_code("firmware/stm32f4_blink/src/meta_head.c", 51, 81)  # meta_forward + meta_predict

**`firmware/stm32f4_blink/src/meta_head.c`** — lignes 51–81

float meta_forward(const MetaHead *m, const float *feats)
{
#if META_HIDDEN > 0
    /* MLP : relu(W1·x + b1) → sigmoid(w2·h + b2) */
    float hidden[META_HIDDEN];   /* MEM: META_HIDDEN × 4 B @ FP32 (stack) */
    for (int j = 0; j < META_HIDDEN; j++) {
        float acc = m->b1[j];
        for (int i = 0; i < META_N_FEATURES; i++) {
            acc += m->w1[j][i] * feats[i];
        }
        hidden[j] = meta_relu(acc);
    }
    float z = m->b2;
    for (int j = 0; j < META_HIDDEN; j++) {
        z += m->w2[j] * hidden[j];
    }
    return meta_sigmoid(z);
#else
    /* logreg : sigmoid(w·x + b) */
    float z = m->b;
    for (int i = 0; i < META_N_FEATURES; i++) {
        z += m->w[i] * feats[i];
    }
    return meta_sigmoid(z);
#endif
}

int meta_predict(const MetaHead *m, const float *feats)
{
    return meta_forward(m, feats) > 0.5f ? 1 : 0;
}

## S6 — Profiling embarqué : comment les chiffres sont *mesurés*

Le projet refuse tout chiffre inventé. La carte mesure elle-même :

- **Latence** — compteur de cycles **DWT** (`profiling_start()` / `profiling_stop()`), converti
  en µs (180 MHz). C'est la preuve du **Gap 2** (≤ 100 ms) : 5 µs (Maha) → ~650 µs (HDC).
- **RAM (`.bss`)** — calculée via les symboles du *linker script*, renvoyée dans la réponse.
- **Métriques online** (`metrics.c`) — calculées au fil de l'eau, sans recalcul offline :
  accuracy, **AUROC** (fenêtre glissante de 50, Wilcoxon-Mann-Whitney), *forgetting*, RMSE,
  F1-macro.

Ces valeurs voyagent dans la réponse UART (champs `lat_us`, `ram_b`, `acc`, `auroc`, …) et
alimentent directement les tableaux du manuscrit.

In [26]:
show_code("firmware/stm32f4_blink/inc/profiling.h", 1, 40)

**`firmware/stm32f4_blink/inc/profiling.h`** — lignes 1–40

#pragma once
#include <stdint.h>

/* ── Profiling firmware : latence DWT, empreinte .bss, throughput ──────────
 *
 * Usage :
 *   profiling_start()  — arme le compteur DWT avant inférence
 *   profiling_stop()   — capture et stocke durée + RAM
 *   profiling_report() — encode les métriques dans la réponse UART
 *
 * Pas de malloc. Toutes les données dans ProfilingState statique (.bss).
 */

/* MEM: 20 B @ FP32/uint32 en .bss */
typedef struct {
    uint32_t t_start_cycles;    /* DWT CYCCNT au début de l'inférence */
    uint32_t last_latency_us;   /* Dernière latence mesurée en µs */
    uint16_t bss_bytes;         /* Taille .bss calculée au link time */
    uint16_t throughput_ips;    /* Inférences par seconde (glissant) */
    uint32_t inference_count;   /* Compteur total d'inférences */
    uint32_t total_cycles;      /* Cycles accumulés sur la session */
} ProfilingState;

/* Symboles fournis par le linker script (calculés au link time) */
#ifndef TEST_HOST
extern uint32_t _sbss;   /* Début segment .bss */
extern uint32_t _ebss;   /* Fin segment .bss  */
extern uint32_t _estack; /* Sommet de la pile (ORIGIN(RAM)+LENGTH(RAM)) */
#endif

extern ProfilingState g_profiling;

/* ── Mesure du pic de RAM (stack high-water mark) ─────────────────────────
 *
 * `.bss` (profiling_get_bss_bytes) ne compte PAS la pile. Le chemin HDC alloue
 * p.ex. float hv[HDC_DIM] = 4 Ko sur la pile : le pic RAM réel vaut donc
 *   .data + .bss + pic_de_pile.
 * Le startup peint [_ebss, _estack) avec STACK_PAINT_SENTINEL au boot ; après
 * exécution d'une charge, le plus bas mot écrasé donne la profondeur de pile.
 * Sentinelle improbable dans des données réelles (caveat : une valeur de pile

## S7 — Synthèse

Une **seule carte**, **N modèles**, un **dispatch par nibble**. Le PC entraîne et exporte ;
la carte infère et, pour EWC / Mahalanobis / HDC / la tête OtO, **s'adapte en ligne**.

In [27]:
synth = pd.DataFrame([
 ["EWC",        "PC → export",            "ewc_forward (~50 µs)",   "SGD+Fisher en ligne ✅", "exacte",  "inf+MAJ ~240–340 µs"],
 ["Mahalanobis","PC → export",            "maha_score (~5 µs)",     "EMA sur μ (Σ⁻¹ figé) ✅", "exacte",  "~5 µs"],
 ["HDC",        "carte (HW-only)",        "hdc_predict (~590 µs)",  "accumulation additive ✅","N/A",     "~590–650 µs"],
 ["TinyOL",     "PC encodeur figé",       "MSE reconstruction",     "tête OtO INT8 ✅",        "N/A",     "≪ 100 ms"],
 ["Méta",       "PC → export",            "meta_forward (néglig.)", "non (poids figés)",      "exacte",  "négligeable"],
], columns=["modèle","entraînement","inférence carte","MAJ en ligne carte","parité","latence (Gap 2)"])
display(synth)

# --- Schéma récap "1 carte, N modèles" ---
fig, ax = plt.subplots(figsize=(12, 5)); ax.axis("off")
ax.add_patch(FancyBboxPatch((4.4,3.6),3.2,1.0, boxstyle="round,pad=0.02,rounding_size=0.08",
             fc="#ff8a65", ec="#333", lw=1.6))
ax.text(6.0,4.1,"pipeline.c\ndispatch par nibble", ha="center", va="center", fontsize=10)
models = [("EWC","0x10"),("HDC","0x20"),("Maha","0xF0"),("TinyOL","0x80"),
          ("DUAL","0x70"),("PAIR","0x90"),("TRIPLE","0xD0")]
xs = np.linspace(0.6,11.4,len(models))
for x,(m,code_) in zip(xs, models):
    ax.add_patch(FancyBboxPatch((x-0.7,1.2),1.4,0.9, boxstyle="round,pad=0.02,rounding_size=0.06",
                 fc="#90caf9", ec="#333", lw=1.2))
    ax.text(x,1.65,f"{m}\n{code_}", ha="center", va="center", fontsize=8.5)
    ax.add_patch(FancyArrowPatch((6.0,3.6),(x,2.1), arrowstyle="-|>", mutation_scale=12,
                 lw=1.1, color="#888", connectionstyle="arc3,rad=0.05"))
ax.set_xlim(0,12); ax.set_ylim(0.9,4.8)
ax.set_title("Une carte, N modèles : dispatch par nibble depuis pipeline.c", fontsize=12)
save(fig, "s7_recap.png"); plt.show()
print("\\nNotebook terminé — figures dans notebooks/figures/firmware_arch/")

,modèle,entraînement,inférence carte,MAJ en ligne carte,parité,latence (Gap 2)
0,EWC,PC → export,ewc_forward (~50 µs),SGD+Fisher en ligne ✅,exacte,inf+MAJ ~240–340 µs
1,Mahalanobis,PC → export,maha_score (~5 µs),EMA sur μ (Σ⁻¹ figé) ✅,exacte,~5 µs
2,HDC,carte (HW-only),hdc_predict (~590 µs),accumulation additive ✅,N/A,~590–650 µs
3,TinyOL,PC encodeur figé,MSE reconstruction,tête OtO INT8 ✅,N/A,≪ 100 ms
4,Méta,PC → export,meta_forward (néglig.),non (poids figés),exacte,négligeable


✓ figure : notebooks/figures/firmware_arch/s7_recap.png
\nNotebook terminé — figures dans notebooks/figures/firmware_arch/


/tmp/ipykernel_36541/2220287742.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  save(fig, "s7_recap.png"); plt.show()
